In [1]:
import cv2
import mediapipe as mp
from keras.models import load_model
import numpy as np
import pickle
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

In [2]:
def xet_thang_thua(lua_chon_1, lua_chon_2):
    if lua_chon_1 == lua_chon_2:
        return "HÒA NHAU!"

    if (lua_chon_1 == "bua" and lua_chon_2 == "keo") or \
       (lua_chon_1 == "keo" and lua_chon_2 == "bao") or \
       (lua_chon_1 == "bao" and lua_chon_2 == "bua"):
        return "NGƯỜI CHƠI 1 THẮNG!"
    else:
        return "NGƯỜI CHƠI 2 THẮNG!"

In [3]:
model = load_model("E:\\ML_project\\models\\model_kbb4.h5")

In [4]:
danh_sach_chu = ["bao","bua","keo"]

In [5]:
def khoang_cach_toa_do(toa_do_tay):
    danh_sach_toa_do = []                                       #tao danh sach luu toa do cac diem thanh mang 2 chieu [[x1,y1,z1],[x2,y2,z2]]
    for lm in hand_lm.landmark:
        toa_do_mot_diem = [lm.x, lm.y, lm.z]
        danh_sach_toa_do.append(toa_do_mot_diem)

    toa_do = np.array(danh_sach_toa_do)                           #chuyen tu mang 1 chieu sang 2 chieu
    co_tay = toa_do[0]                                          #lay x,y,z co tay
    chieu_dai = np.linalg.norm(toa_do[9] - co_tay) + 1e-6       #tinh chieu dai ban tay (dau ngon giua - co tay)

    chuan_hoa = (toa_do - co_tay)/chieu_dai                     #dua ban tay ve vi tri (0,0)

    ngon_tay = [4,8,12,16,20]                                   #tạo độ các đầu ngón tay
    khoang_cach = []
    for i in ngon_tay:
        khoang_cach_co_tay = np.linalg.norm((toa_do[i] - co_tay)/chieu_dai)         #tinh khoảng các từ các đầu ngón tay tới cổ tay
        khoang_cach.append(khoang_cach_co_tay)       
    for i in range(len(ngon_tay)-1):                
        khoang_cach_ngon_tay = np.linalg.norm(toa_do[ngon_tay[i]] - toa_do[ngon_tay[i+1]])/chieu_dai  #tính khoảng cách giữa các đầu ngón tay
        khoang_cach.append(khoang_cach_ngon_tay)
    
    return np.hstack([chuan_hoa.flatten(), khoang_cach]) 

In [6]:
cam = cv2.VideoCapture(0)   #mo cam
BOX_COLOR = (255, 255, 0)
TEXT_COLOR = (255, 0, 0)
khung_hinh = {0: [], 1: []}
lua_chon_chot = {0: None, 1: None}

In [7]:
def ve_khung(hand_lm, w_frames, h_frames):
    x_min, y_min = 1.0, 1.0
    x_max, y_max = 0.0, 0.0
    
    for lm in hand_lm.landmark:
        if lm.x < x_min: x_min = lm.x
        if lm.y < y_min: y_min = lm.y
        if lm.x > x_max: x_max = lm.x
        if lm.y > y_max: y_max = lm.y
        
    x1, y1 = int(x_min * w_frames), int(y_min * h_frames)
    x2, y2 = int(x_max * w_frames), int(y_max * h_frames)
    
    return x1, y1, x2, y2

In [8]:
def du_doan(hand_lm, model, danh_sach_chu):
    them_khoang_cach = khoang_cach_toa_do(hand_lm)    
    du_lieu = np.array([them_khoang_cach])
    du_doan = model.predict(du_lieu, verbose=0)   

    ket_qua_idx = np.argmax(du_doan)    
    phan_tram = np.max(du_doan) 
    ket_qua_chu = danh_sach_chu[ket_qua_idx] 
    
    return ket_qua_idx, ket_qua_chu, phan_tram

In [9]:
def lua_chon(frame_hien_tai, khung_hinh_cua_tay, danh_sach_chu, target_frames=20):
    ket_qua_vua_chot = None
    
    if len(khung_hinh_cua_tay) == 0:                                   
        khung_hinh_cua_tay.append(frame_hien_tai)
    elif len(khung_hinh_cua_tay) < target_frames:                                
        if frame_hien_tai != khung_hinh_cua_tay[-1]:      
            khung_hinh_cua_tay.clear()
        else:
            khung_hinh_cua_tay.append(frame_hien_tai)                
    
    if len(khung_hinh_cua_tay) == target_frames:      
        ket_qua_vua_chot = danh_sach_chu[khung_hinh_cua_tay[5]]
        khung_hinh_cua_tay.clear()
        
    return ket_qua_vua_chot

In [10]:
while True:
    ret, frames = cam.read()    
    if not ret: break

    rgb = cv2.cvtColor(frames, cv2.COLOR_BGR2RGB)   #doi he mau
    results = hands.process(rgb)    
    h_frames, w_frames, c = frames.shape    #lay kich thuoc khung hinh

    if results.multi_hand_landmarks:
        sorted_hands = sorted(results.multi_hand_landmarks, key=lambda hand: hand.landmark[0].x)  
        
        for idx, hand_lm in enumerate(sorted_hands):

            mp_draw.draw_landmarks(frames, hand_lm, mp_hands.HAND_CONNECTIONS)  #ve khung xuong

            
            x1, y1, x2, y2 = ve_khung(hand_lm, w_frames, h_frames)      #ve khung ban tay
            cv2.rectangle(frames, (x1, y1), (x2, y2), BOX_COLOR, 2)

            ket_qua_idx, ket_qua_chu, phan_tram = du_doan(hand_lm, model, danh_sach_chu)    #du doan

            ten_tay = f"Player {idx + 1}"                                              #viet ket qua ra man hinh
            text = f"{ten_tay} - {ket_qua_chu.upper()}: {phan_tram*100:.1f}%"
            cv2.putText(frames, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, TEXT_COLOR, 2)

            ket_qua_chot = lua_chon(ket_qua_idx, khung_hinh[idx], danh_sach_chu, 20)       #quyet dinh ket qua cuoi cung
            
            if ket_qua_chot is not None:
                lua_chon_chot[idx] = ket_qua_chot
                print(f"{ten_tay} : {ket_qua_chot.upper()}")         
                
        if lua_chon_chot[0] is not None and lua_chon_chot[1] is not None:               #xet thang thua
            ket_qua_tran_dau = xet_thang_thua(lua_chon_chot[0], lua_chon_chot[1]) 
            print("\n" + "="*30)
            print(f"P1 ({lua_chon_chot[0]}) VS P2 ({lua_chon_chot[1]})")
            print(f"KẾT QUẢ: {ket_qua_tran_dau}")
            print("="*30 + "\n")
            lua_chon_chot = {0: None, 1: None} # Reset
            
    cv2.imshow("keo bua bao", frames)
    
    if cv2.waitKey(1) & 0xFF == ord('l'):
        break

e:\ML_project\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Player 1 : KEO
Player 1 : KEO
Player 1 : KEO
Player 1 : KEO
Player 1 : KEO
Player 1 : KEO
Player 1 : KEO
Player 1 : KEO


In [11]:
cam.release()
cv2.destroyAllWindows()